# 07a · SHAP de la votación de ML

Este cuaderno comprueba modelos finales de la semilla 42 y explica la **decisión completa de mayoría**
en España y Cuba. No entrena modelos, no repite la optimización y no sustituye las métricas del test.
La figura del artículo utiliza regresión logística en México y votación de ML en España y Cuba,
según el F1-Macro medio del test filtrado. El notebook 07 reúne las explicaciones y genera la figura.

Requiere el entorno del proyecto, sus modelos originales y `es_core_news_sm`. La ejecución verificada
utilizó SHAP 0.49.1, XGBoost 3.2.0 y scikit-learn 1.8.0. Los archivos originales se conservan.


In [1]:
from pathlib import Path
import os
import hashlib
import json
import bisect
import time
import importlib.metadata
import joblib
import sklearn
import xgboost
import numpy as np
import pandas as pd
import shap
import spacy
from nltk.stem import SnowballStemmer

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/resultados_ml_groupcv_tfidf_lr_rf_10k/seleccion_final_ml.csv').is_file()), None)
if REPO is None:
    raise FileNotFoundError('Ejecute este notebook desde la raíz del proyecto o desde notebooks/.')
DATA=REPO/'data'
ML=DATA/'resultados_ml_groupcv_tfidf_lr_rf_10k'
HV=DATA/'resultados_ml_hard_voting_tfidf10k'
OUT_DIR=Path(os.environ.get('SHAP_VOTACION_OUTPUT_DIR',str(DATA/'resultados_interpretabilidad/votacion_ml')))
OUT_DIR.mkdir(parents=True,exist_ok=True)
STAGE=OUT_DIR
SEED=42
PERMUTACIONES=128
RECALCULAR=False
FEATURE_COLS=['n_exc','n_int','n_may','n_emo','n_ris','n_neg','n_elo','n_com','n_pun']
PREPS={'normal':'','stem':'_stem','lemma':'_lemma'}
NOMBRES_RASGOS={'n_exc':'exclamaciones','n_int':'interrogaciones','n_may':'mayúsculas',
    'n_emo':'emojis','n_ris':'risas','n_neg':'negaciones','n_elo':'alargamientos',
    'n_com':'comillas','n_pun':'puntos suspensivos'}
VERSIONES={p:importlib.metadata.version(p) for p in ['numpy','pandas','scikit-learn','xgboost','shap','spacy','nltk']}
print('Versiones:',VERSIONES)
print('Salida:',OUT_DIR)


Versiones: {'numpy': '1.26.4', 'pandas': '3.0.3', 'scikit-learn': '1.8.0', 'xgboost': '3.2.0', 'shap': '0.49.1', 'spacy': '3.7.5', 'nltk': '3.9.4'}
Salida: /Users/taniac/.codex/.chatgpt-projects/g-p-6ae903dd9238819182016272e40165e1/output/resultados/verificacion_shap_ml_votacion/paquete/votacion_ml


## 1. Reproducción de las predicciones

Comprueba las 600 predicciones por variante de los nueve modelos individuales y las tres votaciones.
Los modelos deben reproducir todas las etiquetas; la tolerancia de probabilidades es 1e-6.
La comprobación usa la semilla 42, que corresponde a los modelos explicados.


In [2]:
def cargar_df(variante,prep,split='test'):
    return pd.read_csv(DATA/f'{split}_clean{PREPS[prep]}_{variante}.csv',dtype={'ID':str})

def preparar_X(df):
    x=df[['MESSAGE_CLEAN']+FEATURE_COLS].copy()
    x['MESSAGE_CLEAN']=x['MESSAGE_CLEAN'].fillna('').astype(str)
    x[FEATURE_COLS]=x[FEATURE_COLS].fillna(0)
    return x

def cargar_modelo(variante,modelo,prep):
    ruta=ML/'modelos_finales'/f'{modelo.lower()}_{variante}_{prep}_seed{SEED}.pkl'
    est=joblib.load(ruta)
    clf=est.steps[-1][1]
    if hasattr(clf,'n_jobs'):
        clf.set_params(n_jobs=1)
    return est,ruta

def verificar_predicciones():
    seleccion=pd.read_csv(ML/'seleccion_final_ml.csv')
    referencia=pd.read_csv(ML/'predicciones_test_ml.csv',dtype={'ID':str})
    ref_hv=pd.read_csv(HV/'hard_voting_test_predicciones.csv',dtype={'ID':str})
    resultados=[]
    filas=[]
    for variante in ('mx','es','cu'):
        preds=[]
        ids_ref=None
        for modelo in ('LR','RF','XGBoost'):
            prep=seleccion.query('variante == @variante and modelo == @modelo').iloc[0].preprocessing
            df=cargar_df(variante,prep)
            assert len(df)==600 and df.ID.is_unique
            ref=referencia.query('variante == @variante and modelo == @modelo and seed == @SEED').set_index('ID').loc[df.ID]
            assert np.array_equal(df.IS_IRONIC,ref.y_true)
            est,ruta=cargar_modelo(variante,modelo,prep)
            x=preparar_X(df)
            pred=np.asarray(est.predict(x)).astype(int)
            proba=est.predict_proba(x)[:,list(est.classes_).index(1)]
            discrepancias=int((pred!=ref.y_pred.to_numpy()).sum())
            error_prob=float(np.max(np.abs(proba-ref.prob_ironia.to_numpy())))
            resultados.append({'variante':variante,'modelo':modelo,'seed':SEED,'preprocessing':prep,
                'n_test':len(df),'discrepancias_prediccion':discrepancias,'max_error_probabilidad':error_prob,
                'n_features':int(est.steps[-1][1].n_features_in_),'ruta_modelo':str(ruta),
                'sha256_modelo':hashlib.sha256(ruta.read_bytes()).hexdigest()})
            print(f'{variante}/{modelo}: diferencias={discrepancias}; máximo error de probabilidad={error_prob:.3g}',flush=True)
            preds.append(pred)
            if ids_ref is not None:
                assert list(df.ID)==ids_ref
            ids_ref=list(df.ID)
            filas.append(pd.DataFrame({'variante':variante,'modelo':modelo,'seed':SEED,'ID':df.ID,
                'y_true':df.IS_IRONIC,'y_pred':pred,'prob_ironia':proba}))
        mayoria=(np.stack(preds).sum(axis=0)>=2).astype(int)
        rh=ref_hv.query('variante == @variante and seed == @SEED').set_index('ID').loc[ids_ref]
        errores=int((mayoria!=rh.y_pred.to_numpy()).sum())
        assert np.array_equal(rh.y_true,df.IS_IRONIC)
        resultados.append({'variante':variante,'modelo':'HardVoting','seed':SEED,'n_test':600,
            'discrepancias_prediccion':errores,'max_error_probabilidad':None})
        print(f'{variante}/HardVoting: diferencias={errores}',flush=True)
        filas.append(pd.DataFrame({'variante':variante,'modelo':'HardVoting','seed':SEED,'ID':ids_ref,
            'y_true':df.IS_IRONIC,'y_pred':mayoria}))
    informe=pd.DataFrame(resultados)
    informe.to_csv(STAGE/'auditoria_predicciones.csv',index=False)
    pd.concat(filas).to_csv(STAGE/'predicciones_verificadas_seed42.csv',index=False)
    ok=bool(informe.discrepancias_prediccion.eq(0).all() and informe.max_error_probabilidad.dropna().max()<1e-6)
    (STAGE/'auditoria_resumen.json').write_text(json.dumps({'todos_reproducen_predicciones':ok,
        'modelos_individuales':9,'votaciones':3,'seed':SEED,'n_predicciones':7200,
        'versiones':{'numpy':np.__version__,'pandas':pd.__version__,'sklearn':sklearn.__version__,
                    'xgboost':xgboost.__version__},'entrenamientos':0},ensure_ascii=False,indent=2))
    assert ok,'Las predicciones recibidas no coinciden; revisar antes de calcular SHAP.'

In [3]:
verificar_predicciones()

mx/LR: diferencias=0; máximo error de probabilidad=2.22e-16
mx/RF: diferencias=0; máximo error de probabilidad=5.55e-17
mx/XGBoost: diferencias=0; máximo error de probabilidad=5.96e-08
mx/HardVoting: diferencias=0
es/LR: diferencias=0; máximo error de probabilidad=2.22e-16
es/RF: diferencias=0; máximo error de probabilidad=1.11e-16
es/XGBoost: diferencias=0; máximo error de probabilidad=2.98e-08
es/HardVoting: diferencias=0
cu/LR: diferencias=0; máximo error de probabilidad=2.22e-16
cu/RF: diferencias=0; máximo error de probabilidad=5.55e-17
cu/XGBoost: diferencias=0; máximo error de probabilidad=5.96e-08
cu/HardVoting: diferencias=0


## 2. Explicación de la decisión de mayoría

PermutationSHAP aproxima los valores de Shapley con 128 permutaciones de ida y vuelta por texto.
La función evaluada devuelve **0 o 1 por mayoría de los tres componentes**; las contribuciones no
explican una probabilidad ni se obtienen promediando SHAP de modelos individuales.

Cada jugador léxico es una aparición de una palabra en la base normalizada. Las coaliciones eliminan
esa palabra simultáneamente de sus formas stem/lemma; las formas se calculan en el contexto completo
y permanecen fijas durante la explicación. Se verifica que las formas completas reproduzcan los CSV
originales. TF-IDF y las predicciones se recalculan para cada texto parcial. Los rasgos lingüísticos
no nulos son jugadores adicionales; su ausencia los lleva a cero, de forma independiente del texto.
La referencia es un texto vacío con los nueve conteos en cero. Las formas repetidas no se agrupan.

Este diseño explica las representaciones de entrada: no es una intervención causal sobre el lenguaje.
La comparación entre modelos y variantes es cualitativa por las diferencias de referencias y unidades.
Se analizan los 577 textos de España y los 599 de Cuba del test filtrado común. Cada explicación debe
reconstruir la decisión guardada con error inferior a 1e-8. El resumen global promedia apariciones,
con n como número de textos distintos. La figura solo incluye términos presentes en al menos cinco.

Con SHAP 0.49.1, la precarga evalúa las mismas coaliciones en lotes y exige que la biblioteca no
necesite evaluaciones adicionales; otras versiones utilizan directamente su explicación estándar.
La versión verificada queda documentada en las salidas. [Documentación de PermutationSHAP](https://shap.readthedocs.io/en/latest/generated/shap.PermutationExplainer.html).


In [4]:
def preparar_variante(variante):
    seleccion=pd.read_csv(ML/'seleccion_final_ml.csv').query('variante == @variante')
    base=cargar_df(variante,'normal').reset_index(drop=True)
    preps={r.modelo:r.preprocessing for r in seleccion.itertuples()}
    modelos={m:cargar_modelo(variante,m,preps[m])[0] for m in ('LR','RF','XGBoost')}
    dfs={p:cargar_df(variante,p).reset_index(drop=True) for p in set(preps.values())}
    for p,df in dfs.items():
        assert list(df.ID)==list(base.ID)
        assert np.array_equal(df[FEATURE_COLS],base[FEATURE_COLS])
    textos=base.MESSAGE_CLEAN.fillna('').astype(str).tolist()
    palabras=[t.split() for t in textos]
    formas={}
    if 'stem' in dfs:
        stemmer=SnowballStemmer('spanish')
        stems=[[stemmer.stem(t) for t in ts] for ts in palabras]
        for partes,guardado in zip(stems,dfs['stem'].MESSAGE_CLEAN.fillna('')):
            assert ' '.join(partes)==guardado, 'El stemming no reproduce el CSV original.'
        formas['stem']=stems
    if 'normal' in dfs:
        formas['normal']=palabras
    if 'lemma' in dfs:
        nlp=spacy.load('es_core_news_sm',disable=['parser','ner'])
        lemas=[]
        for i,(texto,doc) in enumerate(zip(textos,nlp.pipe(textos,batch_size=128))):
            guardado=str(dfs['lemma'].iloc[i].MESSAGE_CLEAN)
            salida=' '.join(t.lemma_ for t in doc if not t.is_space)
            assert salida==guardado, f'La lematización no reproduce el CSV original: {variante}/{base.iloc[i].ID}'
            starts=[]
            cursor=0
            for palabra in palabras[i]:
                pos=texto.index(palabra,cursor)
                starts.append(pos)
                cursor=pos+len(palabra)
            grupos=[[] for _ in palabras[i]]
            for t in doc:
                if not t.is_space:
                    k=bisect.bisect_right(starts,t.idx)-1
                    assert k>=0 and t.idx<starts[k]+len(palabras[i][k])
                    grupos[k].append(t.lemma_)
            alineadas=[' '.join(g) for g in grupos]
            assert ' '.join(alineadas).split()==guardado.split()
            lemas.append(alineadas)
        formas['lemma']=lemas
    excluidos=pd.read_csv(DATA/'resultados_comparacion_final/ids_test_coincidentes_con_train.csv',dtype={'ID':str})
    excluidos=set(excluidos.query('variante == @variante').ID)
    indices=np.flatnonzero(~base.ID.isin(excluidos))
    assert len(indices)=={'es':577,'cu':599}[variante]
    referencia=pd.read_csv(DATA/'resultados_ml_hard_voting_tfidf10k/hard_voting_test_predicciones.csv',dtype={'ID':str})
    referencia=referencia.query('variante == @variante and seed == @SEED').set_index('ID')
    return base,palabras,formas,modelos,preps,indices,referencia

class JuegoPalabrasRasgos:
    def __init__(self,base,palabras,formas,modelos,preps,i):
        self.palabras=palabras[i]
        self.formas={p:lista[i] for p,lista in formas.items()}
        self.modelos=modelos
        self.preps=preps
        self.rasgos=base.iloc[i][FEATURE_COLS].to_numpy(dtype=float)
        self.activos=np.flatnonzero(self.rasgos!=0)
        self.n_palabras=len(self.palabras)
        self.n=self.n_palabras+len(self.activos)
        self.cache={}
        self.n_evaluaciones=0

    def __call__(self,z):
        z=np.asarray(z).reshape(-1,self.n)>0.5
        claves=[np.packbits(row).tobytes() for row in z]
        nuevas={key:row for key,row in zip(claves,z) if key not in self.cache}
        if nuevas:
            keys=list(nuevas)
            masks=np.asarray(list(nuevas.values()))
            rasgos=np.zeros((len(keys),len(FEATURE_COLS)))
            rasgos[:,self.activos]=masks[:,self.n_palabras:]*self.rasgos[self.activos]
            x_preps={}
            for prep,tokens in self.formas.items():
                textos=[' '.join(token for token,keep in zip(tokens,row[:self.n_palabras]) if keep)
                        for row in masks]
                x=pd.DataFrame(rasgos,columns=FEATURE_COLS)
                x.insert(0,'MESSAGE_CLEAN',textos)
                x_preps[prep]=x
            votos=[est.predict(x_preps[self.preps[m]]) for m,est in self.modelos.items()]
            y=(np.asarray(votos).sum(axis=0)>=2).astype(float)
            self.cache.update(zip(keys,y))
            self.n_evaluaciones+=len(keys)
        return np.array([self.cache[key] for key in claves])

    def precargar_permutaciones(self,seed,n_permutaciones):
        """Agrupa las mismas coaliciones de SHAP 0.49.1 para reducir llamadas al pipeline.

        La comprobación posterior exige que PermutationExplainer no necesite nuevas
        evaluaciones: no cambia el muestreo, los valores ni la biblioteca explicadora.
        """
        rng=np.random.RandomState(seed)
        inds=np.arange(self.n)
        pendientes=[]
        for _ in range(n_permutaciones):
            rng.shuffle(inds)
            fila=np.zeros(self.n,dtype=bool)
            pendientes.append(fila.copy())
            for k in inds:
                fila[k]=True
                pendientes.append(fila.copy())
            for k in inds:
                fila[k]=False
                pendientes.append(fila.copy())
            if len(pendientes)>=4000:
                self(np.asarray(pendientes))
                pendientes=[]
        if pendientes:
            self(np.asarray(pendientes))

def ejecutar(variante,permutaciones,limite,etiqueta):
    carpeta=STAGE/etiqueta/variante
    carpeta.mkdir(parents=True,exist_ok=True)
    inicio=time.monotonic()
    base,palabras,formas,modelos,preps,indices,referencia=preparar_variante(variante)
    # Orden estable y aleatorio para que el piloto no se limite a los primeros IDs.
    if limite:
        indices=np.random.RandomState(SEED).choice(indices,min(limite,len(indices)),replace=False)
    print(f'{variante}: textos={len(indices)}, permutaciones={permutaciones}',flush=True)
    for contador,i in enumerate(indices,1):
        identificador=base.iloc[i].ID
        ruta=carpeta/f'{identificador}.json'
        if ruta.exists():
            previo=json.loads(ruta.read_text())
            assert previo['permutaciones']==permutaciones and previo['seed']==SEED
            continue
        juego=JuegoPalabrasRasgos(base,palabras,formas,modelos,preps,i)
        salida=float(juego(np.ones((1,juego.n)))[0])
        assert salida==int(referencia.loc[identificador].y_pred)
        valor_base=float(juego(np.zeros((1,juego.n)))[0])
        seed_local=int(hashlib.sha256(f'{SEED}:{variante}:{identificador}'.encode()).hexdigest()[:8],16)
        nombres=juego.palabras+[f'[{NOMBRES_RASGOS[FEATURE_COLS[j]]}]' for j in juego.activos]
        precarga=shap.__version__=='0.49.1'
        if precarga:
            juego.precargar_permutaciones(seed_local,permutaciones)
        n_precargadas=juego.n_evaluaciones
        explicador=shap.PermutationExplainer(juego,np.zeros((1,juego.n)),seed=seed_local,feature_names=nombres)
        explicacion=explicador(np.ones((1,juego.n)),max_evals=(2*juego.n+1)*permutaciones,
            batch_size=10000,error_bounds=True,silent=True)
        sv=explicacion.values[0]
        if precarga:
            assert juego.n_evaluaciones==n_precargadas, 'La precarga no coincide con las coaliciones de SHAP.'
        error=abs(valor_base+sv.sum()-salida)
        assert np.isfinite(sv).all() and error<1e-8
        filas=[]
        for k,(nombre,valor) in enumerate(zip(nombres,sv)):
            filas.append({'posicion':k,'tipo':'palabra' if k<juego.n_palabras else 'rasgo',
                'termino':nombre,'shap':float(valor),
                'error_std':float(explicacion.error_std[0,k]) if explicacion.error_std is not None else None})
        resultado={'variante':variante,'ID':identificador,'modelo':'HardVoting','seed':SEED,
            'y_true':int(base.iloc[i].IS_IRONIC),'y_pred':int(salida),'salida_modelo':salida,
            'valor_base':valor_base,'error_aditividad':error,'n_palabras':juego.n_palabras,
            'n_rasgos_activos':len(juego.activos),'permutaciones':permutaciones,'seed_local':seed_local,
            'n_evaluaciones_unicas':juego.n_evaluaciones,'texto_normal':str(base.iloc[i].MESSAGE_CLEAN),
            'texto_original':str(base.iloc[i].MESSAGE),'contribuciones':filas}
        ruta.write_text(json.dumps(resultado,ensure_ascii=False,indent=2))
        if contador==1 or contador%10==0 or contador==len(indices):
            print(f'{variante}: {contador}/{len(indices)}; {time.monotonic()-inicio:.1f} s; último n={juego.n}',flush=True)
    registros=[json.loads((carpeta/f'{base.iloc[i].ID}.json').read_text()) for i in indices]
    filas=[{'variante':r['variante'],'ID':r['ID'],'modelo':r['modelo'],'seed':r['seed'],**c}
           for r in registros for c in r['contribuciones']]
    pd.DataFrame(filas).to_csv(carpeta/'shap_votacion_por_texto.csv',index=False)
    pd.DataFrame([{k:v for k,v in r.items() if k!='contribuciones'} for r in registros]).to_csv(
        carpeta/'casos_y_aditividad.csv',index=False)
    d=pd.DataFrame(filas)
    g=d.groupby(['tipo','termino'],sort=True).agg(n_textos=('ID','nunique'),n_ocurrencias=('ID','size'),
        shap_medio=('shap','mean'),shap_abs_medio=('shap',lambda s:s.abs().mean())).reset_index()
    g.insert(0,'modelo','HardVoting');g.insert(0,'variante',variante)
    g.to_csv(carpeta/'shap_votacion_global.csv',index=False)
    print(f'{variante}: terminado en {time.monotonic()-inicio:.1f} s',flush=True)

In [5]:
globales=[]
por_texto=[]
casos=[]
for variante in ['es','cu']:
    carpeta=OUT_DIR/variante
    requeridos=['shap_votacion_global.csv','shap_votacion_por_texto.csv','casos_y_aditividad.csv']
    if RECALCULAR or not all((carpeta/n).is_file() for n in requeridos):
        ejecutar(variante,PERMUTACIONES,0,'')
    else:
        print(variante,': reutiliza los resultados guardados; RECALCULAR=True permite regenerarlos.')
    g=pd.read_csv(carpeta/'shap_votacion_global.csv')
    t=pd.read_csv(carpeta/'shap_votacion_por_texto.csv',dtype={'ID':str})
    c=pd.read_csv(carpeta/'casos_y_aditividad.csv',dtype={'ID':str})
    assert len(c)=={'es':577,'cu':599}[variante]
    assert c.error_aditividad.max()<1e-8
    assert c.permutaciones.eq(PERMUTACIONES).all()
    assert np.isfinite(t.shap).all()
    globales.append(g);por_texto.append(t);casos.append(c)
pd.concat(globales).to_csv(OUT_DIR/'shap_votacion_global.csv',index=False)
pd.concat(por_texto).to_csv(OUT_DIR/'shap_votacion_por_texto.csv',index=False)
pd.concat(casos).to_csv(OUT_DIR/'casos_y_aditividad.csv',index=False)
manifest={'seed':SEED,'permutaciones':PERMUTACIONES,'versiones':VERSIONES,
    'salida_explicada':'decisión binaria de la mayoría; no probabilidad',
    'referencia':'texto vacío y nueve rasgos en cero',
    'unidades':'ocurrencias de palabras normalizadas y rasgos no nulos',
    'formas_preprocesadas_fijas':True,'entrenamientos':0}
(OUT_DIR/'configuracion_shap_votacion.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
display(pd.concat(casos).groupby('variante').agg(n_textos=('ID','size'),
    max_error_aditividad=('error_aditividad','max')))


es : reutiliza los resultados guardados; RECALCULAR=True permite regenerarlos.
cu : reutiliza los resultados guardados; RECALCULAR=True permite regenerarlos.


          n_textos  max_error_aditividad
variante                                
cu             599                   0.0
es             577                   0.0

## 3. Salidas y alcance

- `auditoria_predicciones.csv`: comparación de etiquetas y probabilidades con las salidas guardadas.
- `shap_votacion_por_texto.csv`: contribuciones léxicas y lingüísticas por caso.
- `shap_votacion_global.csv`: resúmenes por término o rasgo, variante y número de textos.
- `casos_y_aditividad.csv`: decisiones, referencias, semillas, número de evaluaciones y errores.
- `configuracion_shap_votacion.json`: configuración y versiones.

`RECALCULAR=False` conserva los análisis completos ya disponibles y comprueba sus dimensiones y
aditividad. El análisis es exploratorio: no estima causas del F1-Macro ni demuestra comprensión
pragmática. SHAP de los modelos individuales se comprueba por separado en el cuaderno de validación.
